In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pandas.plotting import scatter_matrix

plt.style.use("seaborn-v0_8")
%matplotlib inline

## Loading the dataset

In [ ]:
data = pd.read_csv("./old_cars - old_cars.csv")
data.shape

In [ ]:
data.head(10)

In [ ]:
data.info()

## 2) Data cleaning + typing

In [ ]:
# standardizing column names (for type safety)
data.columns = [c.strip() for c in data.columns]

# Convert numeric columns
numeric_columns = ["MPG","Displacement","Horsepower","Weight","Model"]
for c in numeric_columns:
  data[c] = pd.to_numeric(data[c], errors="coerce")
  
  
# Treat 0 horsepower as missing (dataset convention)
data.loc[data["Horsepower"] == 0, "Horsepower"] = np.nan

# Clean up the categories
data["Origin"] = data["Origin"].astype(str).str.strip()
data["Car"] = data["Car"].astype(str).str.strip()

data.info()

In [ ]:
data.isna().sum().sort_values(ascending=False)

In [ ]:
data.describe(include="all").T

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()

cols = ["MPG", "Displacement", "Horsepower", "Weight"]
for ax, c in zip(axes, cols):
  ax.hist(data[c].dropna(), bins=20, color="#4C72B0", edgecolor="white")
  ax.set_title(f"Histogram: {c}")
  ax.set_xlabel(c)
  ax.set_ylabel("Count")

plt.tight_layout()
plt.show()

## Task 1: Grouped bar chart MPG distribution by Origin (2 MPG increments)

**Requirement:** Discretize MPG values into **2-MPG** bins. For each MPG bin, show **3 bars** (US, Europe, Japan) so distributions compare across origins.

**Approach:**
- Build bins from `min(MPG)` to `max(MPG)` in steps of 2
- Use `pd.cut` to assign each car to a bin
- Count cars per bin per origin
- Plot grouped bars with offsets

In [ ]:
mpg = data["MPG"].dropna()

# Build 2-MPG increment bins
min_mpg = np.floor(mpg.min() / 2) * 2
max_mpg = np.ceil(mpg.max() /2) * 2
bins = np.arange(min_mpg,max_mpg + 2, 2)

data_task1 = data.dropna(subset=["MPG","Origin"]).copy()
data_task1["MPG_bin"] = pd.cut(data_task1["MPG"],bins=bins,right=False)

# Count  per bin and origin
counts = (
  data_task1.groupby(["MPG_bin","Origin"])
  .size()
  .unstack(fill_value=0)
)

# Ensure consistent origin order
origin_order = [o for o in ["US","Europe","Japan"] if o in counts.columns]
counts = counts.reindex(columns=origin_order)

counts.head()

In [ ]:
x = np.arange(len(counts.index))
bar_width = 0.25

fig, ax = plt.subplots(figsize=(16, 6))

colors = {"US": "#4C72B0", "Europe": "#55A868", "Japan": "#C44E52"}

for i, origin in enumerate(origin_order):
  ax.bar(
    x + (i - (len(origin_order) - 1) / 2) * bar_width,
    counts[origin].values,
    width=bar_width,
    label=origin,
    color=colors.get(origin, None),
    edgecolor="white",
  )

ax.set_title("Task 1 — MPG distribution (2-MPG bins) by Origin")
ax.set_xlabel("MPG bin (2-MPG increments)")
ax.set_ylabel("Number of cars")

# Make readable bin labels
bin_labels = [f"{int(iv.left)}–{int(iv.left+2)}" for iv in counts.index]
ax.set_xticks(x)
ax.set_xticklabels(bin_labels, rotation=45, ha="right")

ax.legend(title="Origin")
plt.tight_layout()
plt.show()